# Forward curves (rates and commodity-style projections)

Deep-dive reference: **forward rate curves** indexed by time for a fixed **tenor** (e.g. 3M SOFR).

## Concept

A **forward curve** stores projected **simple/curve forward levels** at future dates for an index with a fixed accrual tenor (e.g. 0.25 years for 3M). In rates, this supports floating coupons and curve-consistent projections. For commodities, the same object shape can represent **expected forward prices** when you interpret knots as price levels rather than interest rates.

## API walkthrough

`ForwardCurve(id, tenor, knots, base_date=..., day_count=...)` — use `rate(t)` for the forward level at year fraction `t`.

In [1]:
from datetime import date

from finstack.core.market_data import ForwardCurve

fwd = ForwardCurve(
    "USD-SOFR",
    0.25,
    [(0.0, 0.04), (1.0, 0.045), (5.0, 0.05)],
    base_date=date(2024, 1, 1),
    day_count="act_360",
)
print("curve:", fwd)
print("projected 3M index fix at 1Y horizon: rate(1.0) =", fwd.rate(1.0))
for t in (0.0, 0.5, 1.0, 3.0, 5.0):
    print(f"  t={t}y -> forward rate {fwd.rate(t):.6f}")

curve: ForwardCurve(id="USD-SOFR")
projected 3M index fix at 1Y horizon: rate(1.0) = 0.045
  t=0.0y -> forward rate 0.040000
  t=0.5y -> forward rate 0.042500
  t=1.0y -> forward rate 0.045000
  t=3.0y -> forward rate 0.047500
  t=5.0y -> forward rate 0.050000


Link to a **discount curve**: over an interval \([t_1, t_2]\), the continuously compounded forward implied by discount factors is \(\ln(DF(t_1)/DF(t_2))/(t_2-t_1)\). That is a different object than the **index forward curve** (fixed tenor), but the two are related in full curve-building pipelines.

In [2]:
from datetime import date

from finstack.core.market_data import DiscountCurve, ForwardCurve

disc = DiscountCurve(
    "USD-OIS",
    date(2024, 1, 1),
    [(0.0, 1.0), (1.0, 0.96), (2.0, 0.92)],
    day_count="act_365f",
)
t1, t2 = 1.0, 2.0
fwd_from_df = disc.forward_rate(t1, t2)
print(f"Discount-implied continuous forward {t1}y -> {t2}y: {fwd_from_df:.6f}")

sofr_3m = ForwardCurve(
    "USD-SOFR-3M",
    0.25,
    [(0.0, 0.042), (1.0, 0.044), (2.0, 0.046)],
    base_date=date(2024, 1, 1),
    day_count="act_360",
)
print(f"SOFR-3M forward curve at t={t1}y: {sofr_3m.rate(t1):.6f} (index projection, not DF parity check)")

Discount-implied continuous forward 1.0y -> 2.0y: 0.042560
SOFR-3M forward curve at t=1.0y: 0.044000 (index projection, not DF parity check)


## Practical example

**Commodity use case:** interpret knots as forward **prices** (USD/bbl) at future times — same API, different economic meaning.

In [3]:
from datetime import date

from finstack.core.market_data import ForwardCurve

# Illustrative forward "prices" along time; rate(t) is the curve level at t.
com_fwd = ForwardCurve(
    "WTI-PROJ",
    1.0 / 12.0,
    [(0.0, 74.0), (0.5, 73.0), (1.0, 72.0), (2.0, 71.0)],
    base_date=date(2024, 1, 1),
    day_count="act_365f",
)
print("Commodity-style forward levels (interpreted as $/bbl):")
for t in (0.0, 0.25, 1.0):
    print(f"  at {t}y: {com_fwd.rate(t):.2f}")

Commodity-style forward levels (interpreted as $/bbl):
  at 0.0y: 74.00
  at 0.25y: 73.50
  at 1.0y: 72.00


## Takeaways

- **`tenor`** fixes the index accrual length (e.g. `0.25` for 3M); **`rate(t)`** is the forward **level** at horizon `t`.
- **Discount curves** give **implied forwards** via `forward_rate`; **forward curves** are the usual store for **floating index** projections — joint calibration aligns them in production.
- **Commodities:** you can reuse the type for forward **price** term structures; dedicated **`PriceCurve`** is often clearer when you are not modeling a rate index.

## Multi-curve calibration (OIS discount + index forward)

Production rates desks maintain separate curves for discounting (OIS) and index projection (e.g. 3M SOFR). A **multi-step calibration plan** bootstraps the discount curve first, then uses it as the discounting backbone for the forward curve step.

The forward step references the discount curve via `"discount_curve_id"` — the engine automatically feeds the discount curve produced in step 1 into step 2.

The quote strip below is intentionally **internally consistent** and includes a **3Y forward-swap anchor** so the long end remains well behaved. The example also **fails closed**: if calibration does not succeed, it raises immediately instead of printing misleading forward levels.

In [4]:
import json

from finstack.valuations import calibrate

def tenor(count, unit):
    return {"tenor": {"count": count, "unit": unit}}

plan = {
    "schema": "finstack.calibration/2",
    "plan": {
        "id": "usd-multi-curve",
        "description": "OIS discount + 3M SOFR forward curve",
        "quote_sets": {
            "ois_quotes": [
                {"class": "rates", "type": "deposit", "id": "OIS-DEP-1M", "index": "USD-SOFR", "pillar": tenor(1, "months"), "rate": 0.0525},
                {"class": "rates", "type": "swap", "id": "OIS-SWP-1Y", "index": "USD-SOFR-OIS", "pillar": tenor(1, "years"), "rate": 0.0540},
                {"class": "rates", "type": "swap", "id": "OIS-SWP-2Y", "index": "USD-SOFR-OIS", "pillar": tenor(2, "years"), "rate": 0.0480},
                {"class": "rates", "type": "swap", "id": "OIS-SWP-5Y", "index": "USD-SOFR-OIS", "pillar": tenor(5, "years"), "rate": 0.0420},
            ],
            "fwd_quotes": [
                {"class": "rates", "type": "futures", "id": "SOFR-FUT-M1", "contract": "CME:SR3", "expiry": "2025-03-19", "price": 94.60, "convexity_adjustment": 0.0002},
                {"class": "rates", "type": "futures", "id": "SOFR-FUT-M2", "contract": "CME:SR3", "expiry": "2025-06-18", "price": 94.70, "convexity_adjustment": 0.0003},
                {"class": "rates", "type": "swap", "id": "FWD-SWP-1Y", "index": "USD-SOFR-3M", "pillar": tenor(1, "years"), "rate": 0.0530},
                {"class": "rates", "type": "swap", "id": "FWD-SWP-2Y", "index": "USD-SOFR-3M", "pillar": tenor(2, "years"), "rate": 0.0500},
                {"class": "rates", "type": "swap", "id": "FWD-SWP-3Y", "index": "USD-SOFR-3M", "pillar": tenor(3, "years"), "rate": 0.0490},
            ],
        },
        "steps": [
            {
                "id": "USD-OIS",
                "quote_set": "ois_quotes",
                "kind": "discount",
                "curve_id": "USD-OIS",
                "currency": "USD",
                "base_date": "2025-01-15",
                "method": "Bootstrap",
                "interpolation": "linear",
            },
            {
                "id": "USD-SOFR-3M",
                "quote_set": "fwd_quotes",
                "kind": "forward",
                "curve_id": "USD-SOFR-3M",
                "currency": "USD",
                "base_date": "2025-01-15",
                "tenor_years": 0.25,
                "discount_curve_id": "USD-OIS",
                "method": "Bootstrap",
                "interpolation": "linear",
            },
        ],
        "settings": {"use_parallel": False},
    },
}

result = calibrate(json.dumps(plan))
if not result.success:
    raise RuntimeError(result.report_to_dataframe().to_string(index=False))
print("Success:", result.success)
print()

disc = result.market.get_discount("USD-OIS")
fwd = result.market.get_forward("USD-SOFR-3M")

print("Tenor (y)   OIS DF       OIS zero     3M fwd rate")
for t in [0.25, 0.5, 1.0, 2.0, 3.0, 5.0]:
    print(f"  {t:5.2f}     {disc.df(t):.8f}  {disc.zero(t):.6f}   {fwd.rate(t):.6f}")

Success: True

Tenor (y)   OIS DF       OIS zero     3M fwd rate
   0.25     0.98687157  0.052861   0.053800
   0.50     0.97381036  0.053077   0.053201
   1.00     0.94768795  0.053730   0.046846
   2.00     0.90927214  0.047555   0.045499
   3.00     0.87710976  0.043708   0.046524
   5.00     0.81298497  0.041409   0.048733


In [5]:
# Per-step calibration quality
print(result.report_to_dataframe().to_string(index=False))

    step_id  success  iterations  max_residual         rmse                                                                          convergence_reason
    USD-OIS     True         321  8.333882e-13 6.508162e-13 generic bootstrap calibration converged: max_residual (8.33e-13) within tolerance (1.00e-8)
USD-SOFR-3M     True         217  4.145295e-13 2.555749e-13 generic bootstrap calibration converged: max_residual (4.15e-13) within tolerance (1.00e-8)
